# Real-Time Systems: Embedded AI and Edge Measurement
### *Modern Strain Gage Technology* — Companion Notebook

Companion notebook to Chapter 61 — Real-Time Systems: Embedded AI and Edge Measurement.

**Structural health monitoring at scale means computation at the sensor edge.** When a bridge carries hundreds of gages and the radio link carries only a trickle, the intelligence has to move to where the physics is happening.

A bridge with 500 gages sampling at 1 kHz generates 500,000 samples per second — data that cannot all be transmitted wirelessly on available bandwidth, stored indefinitely, or processed retrospectively for real-time alarm detection. *Edge computing* moves the inference to the sensor itself: a microcontroller extracts features, runs a compact model, and transmits only the alert or summary statistic rather than the raw waveform.

The constraints are severe. An ARM Cortex-M4 running at 168 MHz has roughly 210 million 32-bit operations per second, 256 KB of flash, and 64 KB of RAM. A gradient-boosted regressor with 100 shallow (depth-3) trees needs only a few kilobytes of node tables in C, but deeper trees grow exponentially and a 300-tree, depth-8 ensemble can exhaust the flash. Feature extraction — computing RMS, kurtosis, spectral centroid from a 500-sample window — takes on the order of a hundred microseconds on the microcontroller — cheap, but only if the code is written for it. Meeting the real-time deadline (the time between windows) requires careful algorithm selection, not just implementation quality.

The **CUSUM** algorithm is particularly well-suited to edge deployment: it maintains only two floating-point numbers of state, updates with a single addition and comparison per sample, and detects gradual shifts more reliably than a fixed threshold. This combination of simplicity and sensitivity makes it a standard algorithm in edge structural health monitoring deployments.

**This notebook demonstrates:**
1. Multi-feature sliding window — RMS, kurtosis, peak-to-peak, and zero-crossing rate across three injected event types (overload, crack spikes, frequency shift)
2. CUSUM change detection — earlier detection of gradual shifts than a fixed 3σ threshold, with per-event alarm timing comparison
3. Compute budget — feature extraction and inference latency on host CPU with projected Cortex-M4 timing
4. Edge model tradeoff — accuracy vs. inference latency for seven models, from linear regression to a 300-tree GBR

In [ ]:
# Colab setup — uncomment to install dependencies in a fresh runtime:
# !pip install numpy matplotlib scikit-learn

from pathlib import Path
import time

import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import rfft

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

# Reproducibility. Every estimator and RNG in this notebook is seeded from this
# constant so results are identical run-to-run. Sections that build their own
# synthetic data reseed locally (with a fixed value) so each is deterministic
# even when run in isolation.
RANDOM_STATE = 0

# Figures are written here. Do NOT rename these files — the book includes them.
OUT_DIR = Path("../src/media/ch61")
OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 1 — Multi-Feature Sliding Window

A single feature like RMS is a blunt instrument. A crack produces impulsive, high-kurtosis bursts. Loosening hardware shows up as a rising zero-crossing rate. Overload events push peak-to-peak through a structural limit. Each failure mode leaves a different fingerprint, and a real-time system that computes only RMS will miss anything that doesn't raise average power.

The sliding window is the fundamental unit of edge computation: a fixed-length buffer advances in steps, and a small feature vector is extracted from each buffer position. The window length sets temporal resolution; the step size sets update rate. Together they define the latency between an event occurring and the system detecting it.

This cell simulates a 30-second strain record with three overlapping events injected at different times:
- **t = 8–10 s**: a gradual overload (amplitude increase — visible in RMS and pk-pk)  
- **t = 16–17 s**: an impulsive crack event (spike train — visible in kurtosis)  
- **t = 22–24 s**: a frequency shift (structural loosening — visible in zero-crossing rate)

In [ ]:
# ── 1. Multi-Feature Sliding Window ─────────────────────────────────────────
# Section constants (kept local so this section is deterministic on its own).
FS = 1000.0            # sampling rate (Hz)
DURATION_S = 30.0      # record length (s)
BASELINE_S = 6.0       # quiet interval at the start, used to learn thresholds
SIGMA_MULT = 3.0       # alarm threshold = baseline mean + SIGMA_MULT * baseline std
WINDOW_S = 0.5         # sliding-window length (s)
STEP_S = 0.1           # sliding-window hop between successive windows (s)

np.random.seed(3)      # fixed seed -> the injected noise (and this figure) is reproducible
t = np.arange(0, DURATION_S, 1 / FS)
sig = 200 + 30 * np.sin(2 * np.pi * 0.5 * t) + np.random.normal(0, 8, len(t))

# Event 1: gradual overload, t = 8–10 s (raises RMS and peak-to-peak)
m1 = (t >= 8) & (t < 10)
sig[m1] += 120 * np.sin(2 * np.pi * 0.5 * t[m1])

# Event 2: impulsive crack spikes, t = 16–17 s (raises kurtosis)
m2 = (t >= 16) & (t < 17)
spike_times = np.arange(16, 17, 0.08)
for st in spike_times:
    pulse = np.exp(-((t - st) ** 2) / (2 * (0.005) ** 2))
    sig += 350 * pulse * m2

# Event 3: frequency shift / loosening, t = 22–24 s (raises zero-crossing rate)
m3 = (t >= 22) & (t < 24)
sig[m3] += 25 * np.sin(2 * np.pi * 18 * t[m3])


def window_features(w, fs):
    """Return (rms, kurtosis, peak_to_peak, zero_crossing_rate) for one window.

    kurtosis is the non-excess fourth standardized moment (about 3 for Gaussian
    noise); zero_crossing_rate counts sign changes about the window mean and is
    reported in Hz. `s == 0` (a constant window) is guarded to avoid a divide.
    """
    m = w.mean()
    s = w.std()
    rms = np.sqrt(np.mean(w ** 2))
    kurt = np.mean((w - m) ** 4) / s ** 4 if s > 0 else 0.0
    pkpk = w.max() - w.min()
    zcr = np.sum(np.diff(np.sign(w - m)) != 0) / (len(w) / fs)
    return rms, kurt, pkpk, zcr


# Sliding-window feature extraction
win = int(WINDOW_S * FS)     # 500-sample window
step = int(STEP_S * FS)      # 100-sample hop

t_w, rms_w, kurt_w, pkpk_w, zcr_w = [], [], [], [], []
for i in range(0, len(sig) - win, step):
    rms, kurt, pkpk, zcr = window_features(sig[i:i + win], FS)
    rms_w.append(rms)
    kurt_w.append(kurt)
    pkpk_w.append(pkpk)
    zcr_w.append(zcr)
    t_w.append(t[i + win // 2])

t_w = np.array(t_w)
rms_w = np.array(rms_w)
kurt_w = np.array(kurt_w)
pkpk_w = np.array(pkpk_w)
zcr_w = np.array(zcr_w)

# Baselines from the first BASELINE_S seconds -> per-feature SIGMA_MULT-sigma thresholds
base = t_w < BASELINE_S
thresh = {
    'RMS':  rms_w[base].mean()  + SIGMA_MULT * rms_w[base].std(),
    'Kurt': kurt_w[base].mean() + SIGMA_MULT * kurt_w[base].std(),
    'PkPk': pkpk_w[base].mean() + SIGMA_MULT * pkpk_w[base].std(),
    'ZCR':  zcr_w[base].mean()  + SIGMA_MULT * zcr_w[base].std(),
}

events = [(8, 10, 'Overload'), (16, 17, 'Crack spikes'), (22, 24, 'Freq. shift')]
feat_series = [('RMS (µε)', rms_w, thresh['RMS'], 'steelblue'),
               ('Kurtosis', kurt_w, thresh['Kurt'], 'tomato'),
               ('Peak-to-peak (µε)', pkpk_w, thresh['PkPk'], 'seagreen'),
               ('Zero-crossing rate (Hz)', zcr_w, thresh['ZCR'], 'goldenrod')]

fig, axes = plt.subplots(5, 1, figsize=(13, 14), sharex=True)

# Raw signal
axes[0].plot(t, sig, color='dimgray', lw=0.6)
for t0, t1, label in events:
    axes[0].axvspan(t0, t1, alpha=0.12, color='tomato')
    axes[0].text((t0 + t1) / 2, sig.max() * 0.97, label, ha='center', fontsize=8, color='tomato')
axes[0].set_ylabel('Strain (µε)')
axes[0].set_title('Raw strain signal (three injected events)')
axes[0].grid(True, linestyle=':', alpha=0.4)

for ax, (label, vals, thr, color) in zip(axes[1:], feat_series):
    alerts = vals > thr
    ax.plot(t_w, vals, color=color, lw=1.4)
    ax.axhline(thr, color='black', lw=0.9, ls='--', alpha=0.6, label=f'{SIGMA_MULT:.0f}σ threshold')
    ax.scatter(t_w[alerts], vals[alerts], color='tomato', s=25, zorder=5)
    for t0, t1, _ in events:
        ax.axvspan(t0, t1, alpha=0.08, color='tomato')
    ax.set_ylabel(label)
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.4)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig61_nb1_sliding_window_features.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 2 — CUSUM Change Detection

A fixed threshold set at μ + 3σ is reactive: it only fires when a single window exceeds the bar. Gradual damage — a slowly propagating crack, a fastener working loose over thousands of cycles — may never push any single window over threshold even as the process mean drifts steadily upward.

The **Cumulative Sum (CUSUM)** control chart was designed for exactly this problem. Instead of asking "is this window unusual?", it asks "has the process been consistently unusual for a while?" It accumulates small deviations from a reference, and sounds the alarm when the accumulated sum crosses a threshold *h*. A slack parameter *k* (typically 0.5σ) absorbs ordinary fluctuations so that noise alone does not drive the sum upward.

Upper-sided CUSUM (detecting upward shifts):

$$S_t = \max\bigl(0,\ S_{t-1} + (x_t - \mu_0 - k)\bigr)$$

Alarm when $S_t > h$. The sum resets to zero whenever it would go negative, which means sustained normal behaviour erases old memory.

This cell runs both a CUSUM and a simple 3σ threshold on the same RMS series from §1 and measures which one raises the alarm earlier for each injected event.

In [ ]:
# ── 2. CUSUM Change Detection ────────────────────────────────────────────────
CUSUM_K_SIGMA = 0.5    # slack, in units of baseline sigma (absorbs ordinary noise)
CUSUM_H_SIGMA = 4.0    # alarm threshold, in units of baseline sigma
POST_EVENT_S = 2.0     # grace window after an event in which an alarm still counts

mu0  = rms_w[base].mean()
sig0 = rms_w[base].std()
k = CUSUM_K_SIGMA * sig0
h = CUSUM_H_SIGMA * sig0


def cusum_upper(x, mu0, k):
    """Upper-sided CUSUM statistic S_t = max(0, S_{t-1} + (x_t - mu0 - k)).

    Accumulates positive deviations of `x` from reference mean `mu0` after
    subtracting slack `k`, and resets to zero whenever the running sum would go
    negative — so sustained normal behaviour erases old evidence. Returns the
    array of S_t values, same length as `x`.
    """
    S = np.zeros(len(x))
    for i in range(1, len(x)):
        S[i] = max(0.0, S[i - 1] + (x[i] - mu0 - k))
    return S


S = cusum_upper(rms_w, mu0, k)
cusum_alarm  = S > h
thresh_alarm = rms_w > thresh['RMS']


def first_alarm(alarm_mask, t_start, t_end):
    """Timestamp of the first alarm attributable to an event.

    Searches `alarm_mask` inside the interval [t_start, t_end + POST_EVENT_S] and
    returns the time of the earliest True sample, or None if the event was missed.
    """
    in_event = (t_w >= t_start) & (t_w <= t_end + POST_EVENT_S)
    hits = t_w[in_event & alarm_mask]
    return hits[0] if len(hits) else None


print(f"{'Event':18s}  {'Event start':>12}  {'3σ alarm':>10}  {'CUSUM alarm':>12}  {'CUSUM advantage':>16}")
print('-' * 75)
for t0, t1, label in events:
    ta_thresh = first_alarm(thresh_alarm, t0, t1)
    ta_cusum  = first_alarm(cusum_alarm,  t0, t1)
    adv = (f'{(ta_thresh - ta_cusum):.2f} s earlier'
           if ta_thresh and ta_cusum else '—')
    print(f"{label:18s}  {t0:>12.1f}  "
          f"{str(round(ta_thresh,2)) if ta_thresh else 'missed':>10}  "
          f"{str(round(ta_cusum,2))  if ta_cusum  else 'missed':>12}  "
          f"{adv:>16}")

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle('CUSUM vs Fixed Threshold — RMS Feature', fontsize=12)

axes[0].plot(t_w, rms_w, color='steelblue', lw=1.4, label='Sliding RMS')
axes[0].axhline(thresh['RMS'], color='black', lw=1, ls='--', label='3σ threshold')
axes[0].scatter(t_w[thresh_alarm], rms_w[thresh_alarm], color='tomato', s=30, zorder=5, label='3σ alarm')
for t0, t1, lbl in events:
    axes[0].axvspan(t0, t1, alpha=0.1, color='tomato')
axes[0].set_ylabel('RMS (µε)'); axes[0].set_title('Fixed 3σ threshold')
axes[0].legend(fontsize=9); axes[0].grid(True, linestyle=':', alpha=0.4)

axes[1].plot(t_w, S, color='seagreen', lw=1.6, label='CUSUM statistic S_t')
axes[1].axhline(h, color='black', lw=1, ls='--', label=f'Alarm threshold h = {h:.1f}')
axes[1].scatter(t_w[cusum_alarm], S[cusum_alarm], color='tomato', s=30, zorder=5, label='CUSUM alarm')
for t0, t1, _ in events:
    axes[1].axvspan(t0, t1, alpha=0.1, color='tomato')
axes[1].set_ylabel('Cumulative sum')
axes[1].set_title(f'CUSUM (k = {CUSUM_K_SIGMA}σ, h = {CUSUM_H_SIGMA:.0f}σ)')
axes[1].legend(fontsize=9); axes[1].grid(True, linestyle=':', alpha=0.4)

# Detection delay comparison
for t0, t1, label in events:
    ta_thresh = first_alarm(thresh_alarm, t0, t1)
    ta_cusum  = first_alarm(cusum_alarm,  t0, t1)
    for ax in axes[:2]:
        if ta_thresh: ax.axvline(ta_thresh, color='tomato',  lw=1.2, ls=':')
        if ta_cusum:  ax.axvline(ta_cusum,  color='seagreen', lw=1.2, ls=':')

# Alarm timeline (which detector fired, and when)
axes[2].fill_between(t_w, 0, thresh_alarm.astype(float), step='mid',
                     alpha=0.5, color='tomato', label='3σ alarm')
axes[2].fill_between(t_w, 0, cusum_alarm.astype(float)*0.6, step='mid',
                     alpha=0.5, color='seagreen', label='CUSUM alarm')
for t0, t1, lbl in events:
    axes[2].axvspan(t0, t1, alpha=0.1, color='gray')
    axes[2].text((t0+t1)/2, 0.85, lbl, ha='center', fontsize=8)
axes[2].set_yticks([0, 0.6, 1]); axes[2].set_yticklabels(['—', 'CUSUM', '3σ'])
axes[2].set_xlabel('Time (s)'); axes[2].set_title('Alarm timeline')
axes[2].legend(fontsize=9); axes[2].grid(True, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig61_nb1_cusum_detection.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 3 — Compute Budget

Before deploying any algorithm to an edge processor, you must answer a simple question: **does it finish before the next window arrives?**

At 1 kHz sampling with a 500-sample window and 100-sample step, a new window is ready every 100 ms. Every algorithm in the pipeline — feature extraction, normalization, model inference, alarm logic — must complete within that 100 ms budget. Miss the deadline and you drop windows; drop enough windows and you miss events.

This cell times each stage of the pipeline on representative data and builds a budget table. Times are measured on the host CPU in Python, where interpreter and call overhead dominate small operations — a single decision-tree prediction through scikit-learn costs tens of microseconds, almost none of it arithmetic. Scaling those numbers by a "microcontroller slowdown factor" therefore misleads. The Cortex-M4 bars are instead estimated from operation counts for plain C at 168 MHz; they should be confirmed on the target hardware, but they show the right order of magnitude — and that on this budget, computation is not the binding constraint.

In [ ]:
# ── 3. Compute Budget ────────────────────────────────────────────────────────
BUDGET_MS = 100.0                 # real-time deadline: a new window arrives every 100 ms
budget_us = BUDGET_MS * 1000

rng = np.random.default_rng(RANDOM_STATE)
w_sample = rng.normal(200, 30, 500)

# A small synthetic regression problem, used only to time model inference.
X_tiny = rng.normal(0, 1, (300, 4))
y_tiny = X_tiny[:, 0] * 2 + X_tiny[:, 1] ** 2 + rng.normal(0, 0.1, 300)
dt_model  = DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE).fit(X_tiny, y_tiny)
gbr_model = GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE).fit(X_tiny, y_tiny)
x_infer = rng.normal(0, 1, (1, 4))


def time_us(fn, reps=5000):
    """Mean wall-clock time of one call to `fn`, in microseconds, over `reps` reps.

    Uses time.perf_counter (highest-resolution monotonic clock). Intended for
    relative comparison of operations on one host, not as an absolute benchmark.
    """
    t0 = time.perf_counter()
    for _ in range(reps):
        fn()
    return (time.perf_counter() - t0) / reps * 1e6

tasks = [
    ('Mean',                   lambda: w_sample.mean()),
    ('Std dev',                lambda: w_sample.std()),
    ('RMS',                    lambda: np.sqrt(np.mean(w_sample**2))),
    ('Peak-to-peak',           lambda: w_sample.max() - w_sample.min()),
    ('Kurtosis',               lambda: np.mean((w_sample-w_sample.mean())**4)/w_sample.std()**4),
    ('Zero-crossing rate',     lambda: np.sum(np.diff(np.sign(w_sample-w_sample.mean()))!=0)),
    ('FFT (512-pt)',            lambda: rfft(w_sample)),
    ('All 6 features',         lambda: (w_sample.mean(), w_sample.std(),
                                        np.sqrt(np.mean(w_sample**2)),
                                        w_sample.max()-w_sample.min(),
                                        np.mean((w_sample-w_sample.mean())**4)/w_sample.std()**4,
                                        np.sum(np.diff(np.sign(w_sample-w_sample.mean()))!=0))),
    ('Decision tree infer',    lambda: dt_model.predict(x_infer)),
    ('GBR infer (100 trees)',  lambda: gbr_model.predict(x_infer)),
    ('CUSUM update (1 step)',  lambda: max(0, 0 + (w_sample[-1] - 200 - 5))),
]

timings = {}
print(f"{'Operation':38s}  {'Host CPU (µs)':>13}  {'% of budget':>12}  {'Fits?':>6}")
print('-' * 78)
for name, fn in tasks:
    us = time_us(fn)
    timings[name] = us
    pct = us / budget_us * 100
    fits = 'YES' if pct < 5 else ('OK' if pct < 50 else 'TIGHT')
    print(f"{name:38s}  {us:>13.2f}  {pct:>11.4f}%  {fits:>6}")

total_us = timings['All 6 features'] + timings['GBR infer (100 trees)'] + timings['CUSUM update (1 step)']
print(f"\nFull pipeline (6 features + GBR + CUSUM):  {total_us:.1f} µs  "
      f"({total_us/budget_us*100:.3f}% of {BUDGET_MS:.0f} ms budget)")

# ── Visual budget chart ───────────────────────────────────────────────────────
# Cortex-M4 estimate from OPERATION COUNTS, not from scaling the host timings.
# The host numbers above are dominated by Python/NumPy call overhead (a single
# sklearn tree prediction "costs" tens of microseconds, almost all of it
# interpreter), so multiplying them by a slowdown factor says nothing about C on
# a microcontroller. Instead, count cycles for a plain C loop on a 168 MHz
# Cortex-M4F over the N = 500-sample window: ~4-6 cycles per element per pass
# (load, FPU op, loop overhead), ~10 cycles per tree-node comparison, and the
# CMSIS-DSP real FFT at ~12,000 cycles for 512 points. These are estimates to
# be confirmed on the target, but they are the right order of magnitude.
MCU_HZ = 168e6
N_WIN = len(w_sample)
GBR_DEPTH = gbr_model.max_depth            # sklearn default: 3
mcu_cycles = {
    'Mean':                   4 * N_WIN,                  # one pass
    'Std dev':                8 * N_WIN,                  # two passes
    'RMS':                    5 * N_WIN + 14,             # one pass + sqrt
    'Peak-to-peak':           6 * N_WIN,                  # two compares per sample
    'Kurtosis':               14 * N_WIN,                 # mean pass + 4th-moment pass
    'Zero-crossing rate':     6 * N_WIN,                  # sign compare per sample
    'FFT (512-pt)':           12_000,                     # CMSIS-DSP arm_rfft_fast_f32
    'Decision tree infer':    10 * dt_model.get_depth(),  # one compare per level
    'GBR infer (100 trees)':  100 * (10 * GBR_DEPTH + 4), # walk + accumulate per tree
    'CUSUM update (1 step)':  20,                         # add, subtract, max, compare
}
mcu_cycles['All 6 features'] = sum(mcu_cycles[k] for k in
    ('Mean', 'Std dev', 'RMS', 'Peak-to-peak', 'Kurtosis', 'Zero-crossing rate'))

op_names  = list(timings.keys())
host_vals = [timings[n] for n in op_names]
mcu_vals  = [mcu_cycles[n] / MCU_HZ * 1e6 for n in op_names]   # microseconds

fig, ax = plt.subplots(figsize=(13, 6))
y = np.arange(len(op_names))
ax.barh(y - 0.20, host_vals, height=0.38, color='steelblue', alpha=0.85, label='Host CPU, Python/NumPy (measured, µs)')
ax.barh(y + 0.20, mcu_vals,  height=0.38, color='tomato',    alpha=0.75, hatch='///', edgecolor='darkred', label='Cortex-M4 @ 168 MHz, C (op-count estimate, µs)')
ax.axvline(budget_us, color='black', lw=1.8, ls='--', label=f'{BUDGET_MS:.0f} ms deadline ({budget_us:,.0f} µs)')

ax.set_yticks(y)
ax.set_yticklabels(op_names, fontsize=10)
ax.set_xlabel('Execution time (µs, log scale)')
ax.set_xscale('log')
ax.set_xlim(left=0.05)
ax.set_title('Compute budget — measured Python on host vs estimated C on a Cortex-M4', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, axis='x', linestyle=':', alpha=0.4)

# Annotate the feature-extraction bar on the MCU projection
full_idx = op_names.index('All 6 features')
mcu_full = mcu_vals[full_idx]
ax.annotate(f'Features alone:\n~{mcu_full:.0f} µs on MCU\n({mcu_full/budget_us*100:.2f}% of budget)',
            xy=(mcu_full, full_idx + 0.20),
            xytext=(mcu_full * 8, full_idx + 1.8),
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.2),
            fontsize=8.5, color='#c0392b')

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig61_nb1_edge_compute_budget.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 4 — Edge Model Tradeoff: Accuracy vs Latency

A GBR with 200 trees is accurate. It is also slow, memory-hungry, and impossible to fit on a microcontroller with 256 KB of flash. The edge deployment question is not "which model is best?" but "which model is good *enough* within the hardware constraints?"

The tradeoff has two axes:
- **Inference latency** — how long one prediction takes (µs)
- **Prediction accuracy** — RMSE on held-out test data

A model that is 5% less accurate but 100× faster may be the only viable choice on a Cortex-M4. A model that fits in RAM and requires no floating-point unit may be the only choice on an ultra-low-power sensor node.

This cell trains **seven** models of increasing complexity on the same structural load-estimation problem and plots accuracy against both latency and model size. Because the vertical axis is *error*, lower is better, and the useful models lie on the **Pareto frontier** — the non-dominated lower-left boundary of the cloud of points. Reading that frontier off the scatter, rather than picking a single "best" model, is the edge-design skill this section is meant to build.

In [ ]:
# ── 4. Edge Model Tradeoff ────────────────────────────────────────────────────
SEED_DATA = 42
np.random.seed(SEED_DATA)     # fixes the synthetic dataset (and this figure)
N = 1000
eps1 = np.random.uniform(0, 3000, N)
eps2 = np.random.uniform(0, 2000, N)
snap = np.tanh((eps1 - 1800) / 120)
load = (2.1*eps1 + 1200*snap + 0.9*np.where(eps2 > 1200, 1.8*(eps2-1200), 0)
        + np.random.normal(0, 400, N))

X = np.column_stack([eps1, eps2])
X_tr, X_te, y_tr, y_te = train_test_split(X, load, test_size=0.25, random_state=RANDOM_STATE)
x1 = X_te[:1]

candidates = [
    ('Linear regression',    LinearRegression()),
    ('Decision tree d=3',    DecisionTreeRegressor(max_depth=3, random_state=RANDOM_STATE)),
    ('Decision tree d=8',    DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE)),
    ('Random forest  50',    RandomForestRegressor(n_estimators=50,  random_state=RANDOM_STATE)),
    ('Random forest 200',    RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)),
    ('GBR  100 trees',       GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE)),
    ('GBR  300 trees',       GradientBoostingRegressor(n_estimators=300, random_state=RANDOM_STATE)),
]

results = []
for name, model in candidates:
    model.fit(X_tr, y_tr)
    rmse = np.sqrt(mean_squared_error(y_te, model.predict(X_te)))
    lat  = time_us(lambda m=model: m.predict(x1), reps=2000)
    if hasattr(model, 'n_estimators'):
        n_nodes = sum(e.tree_.node_count for e in
                      (model.estimators_.ravel() if hasattr(model.estimators_[0], '__iter__')
                       else model.estimators_))
    elif hasattr(model, 'tree_'):
        n_nodes = model.tree_.node_count
    else:
        n_nodes = X.shape[1] + 1
    results.append((name, rmse, lat, n_nodes))

print(f"{'#':>2}  {'Model':24s}  {'RMSE (N)':>9}  {'Latency (µs)':>13}  {'Nodes':>7}")
print('-' * 62)
for i, (name, rmse, lat, n) in enumerate(results, 1):
    print(f"{i:>2}  {name:24s}  {rmse:>9.1f}  {lat:>13.1f}  {n:>7d}")

markers = ['o', 's', 'D', '^', 'v', 'P', '*']
colors  = ['steelblue', 'seagreen', 'darkorange', 'tomato',
           'firebrick', 'goldenrod', 'purple']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Edge Model Tradeoff — Accuracy vs Inference Latency', fontsize=12)

handles = []
for ax, x_getter, xlabel, xscale, title in [
    (axes[0], lambda r: r[2],
     'Inference latency — single sample (µs)', 'linear',
     'Accuracy vs latency  (lower-left = best edge candidate)'),
    (axes[1], lambda r: r[3],
     'Approx. node count  (proxy for flash/RAM footprint)', 'log',
     'Accuracy vs model size'),
]:
    for i, r in enumerate(results):
        h = ax.scatter(x_getter(r), r[1],
                       s=160, color=colors[i], marker=markers[i], zorder=5,
                       label=f'{i+1}. {r[0]}')
        if ax is axes[0]:
            handles.append(h)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('RMSE (N)', fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.set_xscale(xscale)
    ax.grid(True, linestyle=':', alpha=0.4)

# Single shared legend below both charts — keeps the plot area clear
fig.legend(handles=handles, ncol=4, loc='lower center',
           bbox_to_anchor=(0.5, -0.18),
           fontsize=9, framealpha=0.95, edgecolor='lightgray')

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig61_nb1_edge_model_tradeoff.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Where to take this next

This notebook is a starting point, not a finished system. Three concrete extensions:

1. **Quantize a model and re-measure the budget.** Convert the `GBR 100 trees` model to a fixed-point / int8 representation (for example with `emlearn`, `micromlgen`, or ONNX plus a quantizer) and re-run §3. Quantization is the step the chapter flags as turning a Python-float model into a genuinely *new instrument* — measure how much flash it saves and confirm the §4 RMSE survives the conversion rather than assuming it does.
2. **Trace the CUSUM operating curve.** In §2, sweep the slack `CUSUM_K_SIGMA` and threshold `CUSUM_H_SIGMA` and plot detection latency against false-alarm rate on the quiet baseline. This makes the hard/firm/soft trade-off from the chapter quantitative: exactly how much latency you spend to buy robustness against transient spikes.
3. **Replace the projection with measured hardware timing.** Port the §1 `window_features` extractor to a Cortex-M-class board (an Arduino Nano 33 BLE or a Raspberry Pi Pico) and replace the op-count estimates (`mcu_cycles`) in §3 with real per-operation times. Then check the measured `t_feature` and `t_inference` against the timing-budget equation (Eq. 61.1) in the chapter to see whether the 100 ms deadline actually holds on-device.